# Ppto27 · 10 Bronze (conversor Excel → tabla, en paralelo)
Lee **todas** las plantillas de una carga (Personal, Administrativos, TI, D&A, Impuestos). Spark reparte los archivos entre los
workers y cada uno abre los suyos: el tiempo crece por archivos ÷ núcleos, no en serie. Guarda cada fila de `Tabla1` tal cual (texto).

* **Carga:** por defecto la carpeta más reciente de `10_cargas` (sin contar las `REF_`).
* **Idempotente:** un archivo con el mismo hash que uno ya cargado no se vuelve a leer (queda REPETIDO).

In [ ]:
%pip install openpyxl --quiet

In [ ]:
import sys
sys.path.append('/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27/00_codigo')            # librerías ppto27_*.py subidas al Volume
import time
import ppto27_lib as L, ppto27_spark as SP

cargas = sorted(d.name.rstrip('/') for d in dbutils.fs.ls(L.CARGAS) if d.isDir())
normales = [c for c in cargas if not c.upper().startswith('REF_')]
dbutils.widgets.dropdown('carga', (normales or cargas or [''])[-1], cargas or [''])
CARGA = dbutils.widgets.get('carga')

In [ ]:
t = time.time()
estados, lineas = SP.bronze(spark, f'{L.CARGAS}/{CARGA}', CARGA)
print(f'Carga {CARGA}: {estados} · {lineas:,} líneas · {time.time() - t:.0f} s')
display(spark.sql(f"""
SELECT rubro, pais_carpeta, estado, COUNT(*) AS archivos, SUM(n_lineas) AS lineas
FROM {L.tabla('brz_archivos')} WHERE carga_id = '{CARGA}' GROUP BY ALL ORDER BY rubro, pais_carpeta, estado"""))